# LeetCode 1829 — Maximum XOR for Each Query bằng Python

Link đề: https://leetcode.com/problems/maximum-xor-for-each-query/ · Độ khó: **Medium** · Chủ đề: Array, Bit Manipulation, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def getMaximumXor(self, nums: List[int], maximumBit: int) -> List[int]:
        ...
```

## 1. Mục tiêu học

1. Nắm vững phép **XOR** (`^`): giao hoán, kết hợp, `x ^ x = 0`, `x ^ 0 = x` — và đặc biệt tính chất **"tự hủy"** giúp **bỏ** một phần tử khỏi tổng XOR chỉ bằng cách XOR thêm lần nữa.
2. Hiểu **mặt nạ bit** (bitmask) `mask = 2^m − 1` (toàn bit 1) và vì sao `x ^ mask` là số `k` làm `x ^ k` lớn nhất khi `k < 2^m`.
3. Đi từ vét cạn (thử mọi `k`) → tham lam **từng bit** → công thức `k = X ^ mask` với **XOR tiền tố** cập nhật lùi, tổng `O(n)`.
4. Đọc kỹ thứ tự truy vấn: truy vấn thứ `i` dùng mảng sau khi đã **xóa `i` phần tử cuối**.
5. Kiểm thử đối chiếu nhiều cách trên dữ liệu ngẫu nhiên đúng ràng buộc (mảng **tăng dần**, `nums[i] < 2^maximumBit`).

## 2. Đề bài & ví dụ

Cho mảng `nums` gồm `n` số nguyên **không âm**, **đã sắp tăng dần**, và số nguyên `maximumBit`. Thực hiện `n` lần truy vấn sau:
1. Tìm số nguyên không âm `k < 2^maximumBit` sao cho `nums[0] XOR nums[1] XOR … XOR nums[last] XOR k` **lớn nhất**; `k` là đáp án của truy vấn thứ `i`.
2. **Xóa phần tử cuối** của mảng `nums` hiện tại.

Trả về mảng `answer` với `answer[i]` là đáp án truy vấn thứ `i`.

**Ví dụ 1:** `nums = [0,1,1,3], maximumBit = 2` → `[0,3,2,3]`.
- Truy vấn 1: `nums = [0,1,1,3]`, XOR = `3`, `k = 0` vì `3 XOR 0 = 3` (lớn nhất có thể với 2 bit).
- Truy vấn 2: `nums = [0,1,1]`, XOR = `0`, `k = 3` ⇒ `0 XOR 3 = 3`.
- Truy vấn 3: `nums = [0,1]`, XOR = `1`, `k = 2` ⇒ `1 XOR 2 = 3`.
- Truy vấn 4: `nums = [0]`, XOR = `0`, `k = 3`.

**Ví dụ 2:** `nums = [2,3,4,7], maximumBit = 3` → `[5,2,6,5]`.

**Ví dụ 3:** `nums = [0,1,2,2,5,7], maximumBit = 3` → `[4,3,6,4,6,7]`.

**Ràng buộc:** `1 ≤ n ≤ 10⁵`; `1 ≤ maximumBit ≤ 20`; `0 ≤ nums[i] < 2^maximumBit`; `nums` tăng dần (không giảm).

In [1]:
import random
import time
from functools import reduce
from operator import xor
from itertools import accumulate
from typing import List

examples = [([0, 1, 1, 3], 2, [0, 3, 2, 3]), ([2, 3, 4, 7], 3, [5, 2, 6, 5]), ([0, 1, 2, 2, 5, 7], 3, [4, 3, 6, 4, 6, 7])]
for nums, mb, exp in examples:
    print(f"nums={nums}, maximumBit={mb} -> kỳ vọng {exp}")

nums=[0, 1, 1, 3], maximumBit=2 -> kỳ vọng [0, 3, 2, 3]
nums=[2, 3, 4, 7], maximumBit=3 -> kỳ vọng [5, 2, 6, 5]
nums=[0, 1, 2, 2, 5, 7], maximumBit=3 -> kỳ vọng [4, 3, 6, 4, 6, 7]


## 3. Kiến thức nền

### 3.1 Phép XOR và bảng chân trị

XOR từng bit: `0^0 = 0`, `0^1 = 1`, `1^0 = 1`, `1^1 = 0` — "khác nhau thì 1". Với số nguyên, Python XOR từng cặp bit tương ứng.

In [2]:
for a, b in [(5, 3), (6, 6), (0, 9), (7, 2)]:
    print(f"{a:>2} = {a:04b}  ^  {b:>2} = {b:04b}  ->  {a ^ b:>2} = {a ^ b:04b}")

 5 = 0101  ^   3 = 0011  ->   6 = 0110
 6 = 0110  ^   6 = 0110  ->   0 = 0000
 0 = 0000  ^   9 = 1001  ->   9 = 1001
 7 = 0111  ^   2 = 0010  ->   5 = 0101


### 3.2 Các tính chất quan trọng của XOR

- **Giao hoán, kết hợp:** thứ tự không quan trọng ⇒ "tổng XOR" của một mảng xác định duy nhất.
- **Phần tử trung hòa:** `x ^ 0 = x`.
- **Tự nghịch đảo:** `x ^ x = 0` ⇒ nếu `X = a ^ b ^ c` thì `X ^ c = a ^ b` — **bỏ** `c` khỏi tổng bằng cách XOR thêm `c`. Đây là chìa khóa của bài.

In [3]:
a, b, c = 9, 14, 6
X = a ^ b ^ c
print("X =", X, "| X ^ c =", X ^ c, "== a ^ b =", a ^ b, "| (a^b)^(b^a) =", (a ^ b) ^ (b ^ a))

X = 1 | X ^ c = 7 == a ^ b = 7 | (a^b)^(b^a) = 0


### 3.3 Mặt nạ `2^m − 1`, dịch bit `<<`, lấy bit `>> i & 1`

`(1 << m) − 1` là số có `m` bit thấp đều bằng 1 (ví dụ `m = 3` ⇒ `0b111 = 7`). `x >> i & 1` đọc bit thứ `i` của `x`. `x | (1 << i)` bật bit `i`.

In [4]:
m = 3
mask = (1 << m) - 1
x = 0b101
print("mask =", mask, bin(mask), "| bit của x:", [x >> i & 1 for i in range(m)], "| x ^ mask =", x ^ mask, bin(x ^ mask))

mask = 7 0b111 | bit của x: [1, 0, 1] | x ^ mask = 2 0b10


### 3.4 `functools.reduce(xor, ...)` và `itertools.accumulate(..., xor)`

`reduce(xor, nums)` cho tổng XOR cả mảng; `accumulate(nums, xor)` cho XOR **tiền tố** `p[i] = nums[0] ^ … ^ nums[i]`.

In [5]:
nums = [0, 1, 1, 3]
print("XOR cả mảng:", reduce(xor, nums), "| XOR tiền tố:", list(accumulate(nums, xor)))

XOR cả mảng: 3 | XOR tiền tố: [0, 1, 0, 3]


### 3.5 In số dạng nhị phân: `bin`, `format`, `bit_length`

Khi làm bài bit, **luôn in ra nhị phân** để kiểm tra bằng mắt. `format(x, "05b")` in `x` với đúng 5 chữ số nhị phân (đệm 0 bên trái); `x.bit_length()` cho số bit cần để biểu diễn `x`.

In [6]:
for x in [0, 5, 7, 19]:
    print(f"{x:>3}: bin={bin(x):<8} format 05b={format(x, '05b')} bit_length={x.bit_length()}")

  0: bin=0b0      format 05b=00000 bit_length=0
  5: bin=0b101    format 05b=00101 bit_length=3
  7: bin=0b111    format 05b=00111 bit_length=3
 19: bin=0b10011  format 05b=10011 bit_length=5


### 3.6 Vì sao "bit cao quan trọng hơn mọi bit thấp cộng lại"

`2^i > 2^{i−1} + 2^{i−2} + … + 2^0 = 2^i − 1`. Vì vậy khi so hai số, bit cao nhất khác nhau quyết định số nào lớn hơn — nền tảng của mọi thuật toán **tham lam theo bit** (Trie nhị phân, maximum XOR…).

In [7]:
for i in range(1, 6):
    print(f"2^{i} = {2 ** i:>2} > tổng các bit thấp = {2 ** i - 1}")
print("so sánh 100 (4) và 011 (3):", 0b100 > 0b011)

2^1 =  2 > tổng các bit thấp = 1
2^2 =  4 > tổng các bit thấp = 3
2^3 =  8 > tổng các bit thấp = 7
2^4 = 16 > tổng các bit thấp = 15
2^5 = 32 > tổng các bit thấp = 31
so sánh 100 (4) và 011 (3): True


## 4. Ý tưởng nền tảng / trực giác

**Câu hỏi con:** cho số `X` (với `X < 2^m`), chọn `k < 2^m` để `X ^ k` lớn nhất.

Giá trị lớn nhất có thể của một số `m` bit là `2^m − 1` = toàn bit 1. Muốn `X ^ k` có bit `i` bằng 1 ⇔ `k` có bit `i` **ngược** với `X`. Chọn `k` ngược `X` ở **mọi** bit `0..m−1` ⇒ `X ^ k = 2^m − 1` (đạt trần). Đó chính là `k = X ^ mask` (đảo `m` bit thấp của `X`).

Tính tay **Ví dụ 2** (`m = 3`, `mask = 7 = 111₂`):

| truy vấn | mảng | X (nhị phân) | k = X ^ 7 | X ^ k |
|---|---|---|---|---|
| 1 | [2,3,4,7] | 2^3^4^7 = 2 = 010 | 101 = 5 | 7 |
| 2 | [2,3,4] | 2^3^4 = 5 = 101 | 010 = 2 | 7 |
| 3 | [2,3] | 2^3 = 1 = 001 | 110 = 6 | 7 |
| 4 | [2] | 2 = 010 | 101 = 5 | 7 |

**Quan sát:** mỗi truy vấn chỉ cần **XOR của mảng hiện tại**; khi xóa phần tử cuối `v`, XOR mới = XOR cũ `^ v`.

In [8]:
nums, m, _ = examples[1]
mask = (1 << m) - 1
X = reduce(xor, nums)
for q in range(len(nums)):
    print(f"truy vấn {q+1}: mảng={nums[:len(nums)-q]}, X={X} ({X:03b}), k={X ^ mask} ({X ^ mask:03b})")
    X ^= nums[len(nums) - 1 - q]

truy vấn 1: mảng=[2, 3, 4, 7], X=2 (010), k=5 (101)
truy vấn 2: mảng=[2, 3, 4], X=5 (101), k=2 (010)
truy vấn 3: mảng=[2, 3], X=1 (001), k=6 (110)
truy vấn 4: mảng=[2], X=2 (010), k=5 (101)


## 5. Cách 1 — Brute Force / baseline: tính lại XOR và thử mọi `k`

**Ý tưởng:** đúng như đề: mỗi truy vấn tính XOR của mảng hiện tại (`O(n)`), thử mọi `k ∈ [0, 2^m)` (`O(2^m)`), lấy `k` cho `X ^ k` lớn nhất (với nhiều `k` cùng đạt max, đề không nói — nhưng thực tế max chỉ đạt tại **một** `k`, xem mục 10).

In [9]:
def max_xor_bruteforce(nums, maximumBit):
    arr = list(nums); ans = []
    while arr:
        X = 0
        for v in arr:
            X ^= v
        best_k = max(range(1 << maximumBit), key=lambda k: X ^ k)
        ans.append(best_k)
        arr.pop()
    return ans

for nums, mb, exp in examples:
    got = max_xor_bruteforce(nums, mb); print(got, "kỳ vọng", exp); assert got == exp

[0, 3, 2, 3] kỳ vọng [0, 3, 2, 3]
[5, 2, 6, 5] kỳ vọng [5, 2, 6, 5]
[4, 3, 6, 4, 6, 7] kỳ vọng [4, 3, 6, 4, 6, 7]


**Độ phức tạp:** `O(n · (n + 2^m))` — với `n = 10⁵`, `m = 20` là `10⁵ · (10⁵ + 10⁶) ≈ 10¹¹` ⇒ **quá chậm**.

**Hai nút thắt:**
1. Thử `2^m` giá trị `k` trong khi có thể **xây** `k` trực tiếp.
2. Tính lại XOR từ đầu mỗi truy vấn trong khi chỉ một phần tử thay đổi.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (xây `k` tham lam từng bit):** để `X ^ k` lớn nhất, ưu tiên bit **cao** trước: bit cao bằng 1 có giá trị hơn mọi bit thấp cộng lại (`2^i > 2^i − 1`). Với `k < 2^m`, ta tự do chọn **mọi** bit `0..m−1` của `k` ⇒ chọn bit `i` của `k` là `1 − bit_i(X)` cho mọi `i` ⇒ `k = X ^ mask`, `X ^ k = mask` (lớn nhất tuyệt đối).

**Insight 2 (cập nhật XOR khi xóa):** gọi `X_i` là XOR của mảng ở truy vấn `i`. Truy vấn `i+1` bỏ phần tử `nums[n−1−i]` ⇒ `X_{i+1} = X_i ^ nums[n−1−i]` (tính chất tự nghịch đảo).

**Quy tắc:**
- Nếu bit `i` của `X` là `0` thì bật bit `i` của `k`; nếu là `1` thì để `0`.
- Sau mỗi truy vấn: `X ^= phần tử vừa xóa`.

**Invariant:** trước truy vấn thứ `q` (0-based), `X = nums[0] ^ … ^ nums[n−1−q]`.

**Ghi chú:** mảng "đã sắp" **không** được dùng tới — đó là chi tiết thừa của đề (đôi khi đề cho thêm giả thiết để đánh lạc hướng hoặc vì nguồn gốc bài).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — XOR tiền tố + xây `k` từng bit

**Giải thích từng đoạn:**
1. `X = reduce(xor, nums)` — XOR cả mảng.
2. Với mỗi truy vấn: duyệt bit `i` từ `m−1` xuống `0`, nếu `X` có bit `i` bằng 0 thì `k |= 1 << i`.
3. Ghi `k`, rồi `X ^= nums[cuối]` để chuẩn bị truy vấn sau.

In [10]:
def max_xor_bitwise(nums, maximumBit):
    X = reduce(xor, nums)
    ans = []
    for v in reversed(nums):
        k = 0
        for i in range(maximumBit - 1, -1, -1):
            if (X >> i & 1) == 0:
                k |= 1 << i
        ans.append(k)
        X ^= v
    return ans

print([max_xor_bitwise(n, m) for n, m, _ in examples])

[[0, 3, 2, 3], [5, 2, 6, 5], [4, 3, 6, 4, 6, 7]]


**Code nộp LeetCode (Cách 2):**

In [11]:
class SolutionBitwise:
    def getMaximumXor(self, nums: List[int], maximumBit: int) -> List[int]:
        X = 0
        for v in nums:
            X ^= v
        ans = []
        for v in reversed(nums):
            k = 0
            for i in range(maximumBit):
                if not (X >> i) & 1:
                    k |= 1 << i
            ans.append(k)
            X ^= v
        return ans

print([SolutionBitwise().getMaximumXor(n, m) for n, m, _ in examples])

[[0, 3, 2, 3], [5, 2, 6, 5], [4, 3, 6, 4, 6, 7]]


**Độ phức tạp Cách 2:** `O(n · m)` ≈ `2·10⁶` — chạy tốt, nhưng vòng bit là thừa.

### 7.2 Cách 3 — Công thức `k = X ^ mask`

Vòng bit ở Cách 2 chính là "đảo `m` bit thấp của `X`" = `X ^ mask`. Vì `X < 2^m` (mọi `nums[i] < 2^m` nên XOR của chúng cũng `< 2^m`), phép đảo không chạm bit cao hơn.

In [12]:
def max_xor_mask(nums, maximumBit):
    mask = (1 << maximumBit) - 1
    X = reduce(xor, nums)
    ans = []
    for v in reversed(nums):
        ans.append(X ^ mask)
        X ^= v
    return ans

print([max_xor_mask(n, m) for n, m, _ in examples])

[[0, 3, 2, 3], [5, 2, 6, 5], [4, 3, 6, 4, 6, 7]]


**Code nộp LeetCode (Cách 3):** xem mục 9. **Độ phức tạp:** Time `O(n)`, Space `O(1)` ngoài mảng kết quả.

### 7.3 Cách 4 — XOR tiền tố viết bằng `accumulate`, đáp án đọc ngược

`p = list(accumulate(nums, xor))` ⇒ XOR của mảng ở truy vấn `q` là `p[n−1−q]`. Đáp án là `[p[i] ^ mask for i in reversed(range(n))]` — một dòng, không cần biến cập nhật.

In [13]:
class SolutionPrefix:
    def getMaximumXor(self, nums: List[int], maximumBit: int) -> List[int]:
        mask = (1 << maximumBit) - 1
        p = list(accumulate(nums, xor))
        return [x ^ mask for x in reversed(p)]

print([SolutionPrefix().getMaximumXor(n, m) for n, m, _ in examples])

[[0, 3, 2, 3], [5, 2, 6, 5], [4, 3, 6, 4, 6, 7]]


**Độ phức tạp Cách 4:** Time `O(n)`, Space `O(n)` cho mảng tiền tố. Ưu điểm: thấy rõ "đáp án truy vấn `q` chỉ phụ thuộc tiền tố độ dài `n − q`".

### 7.4 Cách 5 — Trie nhị phân trên tập ứng viên `k` (tổng quát hóa, để hiểu pattern)

Cách 3 dựa vào việc `k` được chọn **tự do** trong `[0, 2^m)`. Nếu `k` chỉ được lấy từ một **tập hữu hạn** (như biến thể ở Bài tập 5), công thức `X ^ mask` không còn dùng được — ta cần **Trie nhị phân**:

1. Chèn mọi ứng viên `k` vào Trie theo bit từ cao xuống thấp (mỗi nút có tối đa 2 con: bit 0 và bit 1).
2. Với mỗi `X`, đi từ gốc: tại bit `i`, **ưu tiên** con có bit **ngược** với bit `i` của `X` (để bit `i` của `X ^ k` bằng 1); nếu không có mới đi con còn lại.

Áp dụng cho bài gốc: tập ứng viên là **toàn bộ** `[0, 2^m)` ⇒ mọi nhánh đều tồn tại ⇒ luôn đi được nhánh ngược ⇒ ra đúng `X ^ mask`. Cài đặt minh họa với `m` nhỏ (Trie đầy đủ có `2^{m+1}` nút — không dùng cho `m = 20` trong thực tế, chỉ để thấy hai cách thống nhất):

In [14]:
def build_trie(values, m):
    root = {}
    for v in values:
        node = root
        for i in range(m - 1, -1, -1):
            node = node.setdefault(v >> i & 1, {})
    return root

def query_trie(root, X, m):
    node, k = root, 0
    for i in range(m - 1, -1, -1):
        want = 1 - (X >> i & 1)
        if want not in node:
            want = 1 - want
        k |= want << i
        node = node[want]
    return k

def max_xor_trie(nums, maximumBit):
    root = build_trie(range(1 << maximumBit), maximumBit)
    X = reduce(xor, nums); ans = []
    for v in reversed(nums):
        ans.append(query_trie(root, X, maximumBit)); X ^= v
    return ans

print([max_xor_trie(n, m) for n, m, _ in examples])

[[0, 3, 2, 3], [5, 2, 6, 5], [4, 3, 6, 4, 6, 7]]


**Độ phức tạp Cách 5:** dựng Trie `O(2^m · m)`, mỗi truy vấn `O(m)`. Với bài gốc thì quá nặng (`2^20 · 20`), nhưng đây là **khuôn mẫu** cho mọi bài "maximize XOR với phần tử thuộc một tập": 421, 1707, 1938, 2935…

## 8. TRACE từng bước

Trace Cách 3 trên **Ví dụ 3** (`nums = [0,1,2,2,5,7]`, `m = 3`, `mask = 111₂`):

| q | mảng hiện tại | X | k = X^111 | phần tử bị xóa | X mới |
|---|---|---|---|---|---|
| 0 | [0,1,2,2,5,7] | 011 (3) | 100 (4) | 7 | 3^7 = 100 (4) |
| 1 | [0,1,2,2,5] | 100 (4) | 011 (3) | 5 | 4^5 = 001 (1) |
| 2 | [0,1,2,2] | 001 (1) | 110 (6) | 2 | 1^2 = 011 (3) |
| 3 | [0,1,2] | 011 (3) | 100 (4) | 2 | 3^2 = 001 (1) |
| 4 | [0,1] | 001 (1) | 110 (6) | 1 | 0 |
| 5 | [0] | 000 (0) | 111 (7) | 0 | 0 |

In [15]:
def trace_mask(nums, m):
    mask = (1 << m) - 1; X = reduce(xor, nums); ans = []
    print(f"mask = {mask:0{m}b}, X ban đầu = {X:0{m}b}")
    for q, v in enumerate(reversed(nums)):
        k = X ^ mask; ans.append(k)
        print(f"q={q}: mảng={nums[:len(nums)-q]}, X={X:0{m}b} ({X}), k={k:0{m}b} ({k}), X^k={X ^ k:0{m}b} | xóa {v} -> X mới={X ^ v:0{m}b}")
        X ^= v
    return ans

assert trace_mask([0, 1, 2, 2, 5, 7], 3) == [4, 3, 6, 4, 6, 7]

mask = 111, X ban đầu = 011
q=0: mảng=[0, 1, 2, 2, 5, 7], X=011 (3), k=100 (4), X^k=111 | xóa 7 -> X mới=100
q=1: mảng=[0, 1, 2, 2, 5], X=100 (4), k=011 (3), X^k=111 | xóa 5 -> X mới=001
q=2: mảng=[0, 1, 2, 2], X=001 (1), k=110 (6), X^k=111 | xóa 2 -> X mới=011
q=3: mảng=[0, 1, 2], X=011 (3), k=100 (4), X^k=111 | xóa 2 -> X mới=001
q=4: mảng=[0, 1], X=001 (1), k=110 (6), X^k=111 | xóa 1 -> X mới=000
q=5: mảng=[0], X=000 (0), k=111 (7), X^k=111 | xóa 0 -> X mới=000


Trace Cách 2 (xây từng bit) cho **một** truy vấn: `X = 2 = 010₂`, `m = 3`:

In [16]:
def trace_bits(X, m):
    k = 0
    for i in range(m - 1, -1, -1):
        b = X >> i & 1
        if b == 0:
            k |= 1 << i
        print(f"  bit {i}: X có {b} -> chọn bit k = {1 - b} | k hiện tại = {k:0{m}b}")
    print(f"  => k = {k}, X ^ k = {X ^ k} ({X ^ k:0{m}b})")
    return k

assert trace_bits(2, 3) == 5

  bit 2: X có 0 -> chọn bit k = 1 | k hiện tại = 100
  bit 1: X có 1 -> chọn bit k = 0 | k hiện tại = 100
  bit 0: X có 0 -> chọn bit k = 1 | k hiện tại = 101
  => k = 5, X ^ k = 7 (111)


## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 3** — `k = X ^ mask`, cập nhật XOR khi xóa phần tử cuối.

In [17]:
class Solution:
    def getMaximumXor(self, nums: List[int], maximumBit: int) -> List[int]:
        mask = (1 << maximumBit) - 1
        X = 0
        for v in nums:
            X ^= v
        ans = []
        for i in range(len(nums) - 1, -1, -1):
            ans.append(X ^ mask)
            X ^= nums[i]
        return ans

for nums, mb, exp in examples:
    got = Solution().getMaximumXor(nums, mb); print(got, exp); assert got == exp

[0, 3, 2, 3] [0, 3, 2, 3]
[5, 2, 6, 5] [5, 2, 6, 5]
[4, 3, 6, 4, 6, 7] [4, 3, 6, 4, 6, 7]


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (`X < 2^m`).** Mọi `nums[i] < 2^m` nghĩa là chúng chỉ có bit trong `0..m−1`; XOR không tạo bit mới ⇒ `X < 2^m`.

**Bổ đề 2 (tối ưu và duy nhất).** Với `0 ≤ k < 2^m`, `X ^ k < 2^m` nên `X ^ k ≤ 2^m − 1 = mask`. Chọn `k = X ^ mask` (cũng `< 2^m`) cho `X ^ k = X ^ X ^ mask = mask` — đạt cận trên. Hơn nữa phép `k ↦ X ^ k` là **song ánh** trên `[0, 2^m)` nên chỉ đúng một `k` cho giá trị `mask` ⇒ đáp án **duy nhất** (khớp với vét cạn lấy `max`).

**Bổ đề 3 (cập nhật).** Invariant: trước truy vấn `q`, `X = ⊕_{j=0}^{n−1−q} nums[j]`. Đúng khi `q = 0`; sau truy vấn, `X ^ nums[n−1−q] = ⊕_{j=0}^{n−2−q} nums[j]` vì `nums[n−1−q] ^ nums[n−1−q] = 0`. ∎

### 10.1 Kiểm chứng bằng thực nghiệm: đáp án duy nhất

Bổ đề 2 nói chỉ đúng **một** `k` đạt `X ^ k = mask`. Ta đếm trực tiếp với mọi `X` khi `m ≤ 6`:

In [18]:
for m in range(1, 7):
    mask = (1 << m) - 1
    counts = {X: sum(1 for k in range(1 << m) if X ^ k == mask) for X in range(1 << m)}
    assert set(counts.values()) == {1}
    assert all(max(range(1 << m), key=lambda k: X ^ k) == X ^ mask for X in range(1 << m))
print("Với mọi m <= 6 và mọi X < 2^m: đúng một k đạt X ^ k = mask, và đó là X ^ mask")

Với mọi m <= 6 và mọi X < 2^m: đúng một k đạt X ^ k = mask, và đó là X ^ mask


## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums) ≤ 10⁵`, `m = maximumBit ≤ 20`.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Vét cạn | `O(n · (n + 2^m))` | `O(n)` (bản sao mảng) |
| 2. XOR cập nhật + từng bit | `O(n · m)` | `O(1)` |
| 3. `k = X ^ mask` | `O(n)` | `O(1)` |
| 4. Tiền tố `accumulate` | `O(n)` | `O(n)` |

In [19]:
rng = random.Random(1829)
m = 20; big = sorted(rng.randrange(1 << m) for _ in range(10 ** 5))
for name, f in [("Cách 2 (từng bit)", SolutionBitwise().getMaximumXor), ("Cách 3 (mask)", Solution().getMaximumXor),
                ("Cách 4 (accumulate)", SolutionPrefix().getMaximumXor)]:
    t = time.perf_counter(); r = f(big, m); print(f"{name:<22} {(time.perf_counter()-t)*1000:7.1f} ms | 3 đáp án đầu {r[:3]}")
small = sorted(rng.randrange(1 << 10) for _ in range(300))
t = time.perf_counter(); max_xor_bruteforce(small, 10); print(f"vét cạn n=300, m=10: {(time.perf_counter()-t)*1000:.0f} ms")

Cách 2 (từng bit)         91.9 ms | 3 đáp án đầu [922897, 125678, 922900]


Cách 3 (mask)             10.9 ms | 3 đáp án đầu [922897, 125678, 922900]
Cách 4 (accumulate)       10.4 ms | 3 đáp án đầu [922897, 125678, 922900]


vét cạn n=300, m=10: 13 ms


## 12. Bộ test

Gồm ví dụ đề, edge case (`n = 1`, toàn 0, toàn bằng `mask`, `m = 1`, `m = 20`), và đối chiếu mọi cách với vét cạn trên mảng ngẫu nhiên **đã sắp**, giá trị `< 2^m` (seed cố định). Kiểm thêm tính chất: mọi `X ^ k` đều bằng `mask`.

In [20]:
ALL = {"trie": lambda n, m: max_xor_trie(n, m) if m <= 10 else max_xor_mask(n, m), "bitwise": max_xor_bitwise, "mask": max_xor_mask, "SolBitwise": SolutionBitwise().getMaximumXor,
       "SolPrefix": SolutionPrefix().getMaximumXor, "submit": Solution().getMaximumXor}
def run_all(nums, mb, with_brute=True):
    res = {k: f(list(nums), mb) for k, f in ALL.items()}
    if with_brute:
        res["brute"] = max_xor_bruteforce(nums, mb)
    vals = {tuple(v) for v in res.values()}
    assert len(vals) == 1, (nums, mb, res)
    ans = res["submit"]
    mask = (1 << mb) - 1
    for q, k in enumerate(ans):                               # kiểm tra tính chất
        X = reduce(xor, nums[:len(nums) - q])
        assert 0 <= k <= mask and X ^ k == mask
    return ans

for nums, mb, exp in examples:
    assert run_all(nums, mb) == exp
edge = [([0], 1, [1]), ([1], 1, [0]), ([0, 0, 0], 3, [7, 7, 7]), ([7, 7], 3, [7, 0]), ([5], 20, [(1 << 20) - 1 - 5]),
        ([1, 1], 1, [1, 0])]
for nums, mb, exp in edge:
    assert run_all(nums, mb) == exp, (nums, mb)
rng = random.Random(18290)
for _ in range(500):
    mb = rng.randint(1, 8); n = rng.randint(1, 25)
    run_all(sorted(rng.randrange(1 << mb) for _ in range(n)), mb)
for _ in range(20):
    mb = rng.randint(15, 20); n = rng.randint(1, 3000)
    run_all(sorted(rng.randrange(1 << mb) for _ in range(n)), mb, with_brute=False)
print("Ví dụ, edge case, 500 test có vét cạn, 20 test lớn (m tới 20): OK")

Ví dụ, edge case, 500 test có vét cạn, 20 test lớn (m tới 20): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `~X` (đảo bit) thay vì `X ^ mask`

In [21]:
def wrong_not(nums, maximumBit):
    X = reduce(xor, nums); ans = []
    for v in reversed(nums):
        ans.append(~X)                                       # SAI: ~X = -X-1, đảo cả vô hạn bit cao
        X ^= v
    return ans

nums, mb, exp = examples[0]
print("sai :", wrong_not(nums, mb), "| đúng:", exp)

sai : [-4, -1, -2, -1] | đúng: [0, 3, 2, 3]


**Vì sao sai:** trong Python số nguyên có "vô hạn" bit dấu; `~X = −X − 1` là số **âm**. Đề yêu cầu `0 ≤ k < 2^m`. **Sửa:** `X ^ mask` (chỉ đảo `m` bit thấp), hoặc `~X & mask`.

### Lỗi 2 — Trả `X ^ k` (giá trị lớn nhất) thay vì `k`

In [22]:
def wrong_return_value(nums, maximumBit):
    mask = (1 << maximumBit) - 1; X = reduce(xor, nums); ans = []
    for v in reversed(nums):
        ans.append(X ^ (X ^ mask))                           # SAI: đây là mask, không phải k
        X ^= v
    return ans

nums, mb, exp = examples[1]
print("sai :", wrong_return_value(nums, mb), "| đúng:", exp)

sai : [7, 7, 7, 7] | đúng: [5, 2, 6, 5]


**Vì sao sai:** đề hỏi **`k`** (số cần XOR thêm), không hỏi giá trị XOR lớn nhất (luôn bằng `mask`). **Sửa:** trả `X ^ mask`.

### Lỗi 3 — Xóa phần tử **đầu** thay vì cuối (đọc sai thứ tự truy vấn)

In [23]:
def wrong_remove_first(nums, maximumBit):
    mask = (1 << maximumBit) - 1; X = reduce(xor, nums); ans = []
    for v in nums:                                           # SAI: duyệt xuôi = xóa phần tử đầu
        ans.append(X ^ mask)
        X ^= v
    return ans

nums, mb, exp = examples[2]
print("sai :", wrong_remove_first(nums, mb), "| đúng:", exp)

sai : [4, 4, 5, 7, 5, 0] | đúng: [4, 3, 6, 4, 6, 7]


**Vì sao sai:** sau mỗi truy vấn đề xóa phần tử **cuối**. Cùng một tập phần tử nhưng thứ tự xóa khác nhau cho các tổng XOR trung gian khác nhau. **Sửa:** duyệt ngược (`reversed(nums)`).

### Lỗi 4 — Quên cập nhật `X` (hoặc cập nhật **trước** khi ghi đáp án)

In [24]:
def wrong_update_before(nums, maximumBit):
    mask = (1 << maximumBit) - 1; X = reduce(xor, nums); ans = []
    for v in reversed(nums):
        X ^= v                                               # SAI: xóa trước khi trả lời
        ans.append(X ^ mask)
    return ans

nums, mb, exp = examples[0]
print("sai :", wrong_update_before(nums, mb), "| đúng:", exp)

sai : [3, 2, 3, 3] | đúng: [0, 3, 2, 3]


**Vì sao sai:** truy vấn đầu tiên dùng **toàn bộ** mảng; xóa trước làm lệch mọi đáp án một bước (và đáp án cuối ứng với mảng rỗng). **Sửa:** ghi đáp án rồi mới `X ^= v`.

### Lỗi 5 — Tính lại XOR từ đầu mỗi truy vấn (đúng nhưng `O(n²)`)

In [25]:
def slow_recompute(nums, maximumBit):
    mask = (1 << maximumBit) - 1
    return [reduce(xor, nums[:len(nums) - q]) ^ mask for q in range(len(nums))]

arr = sorted(random.Random(1).randrange(1 << 16) for _ in range(4000))
for f in (slow_recompute, Solution().getMaximumXor):
    t = time.perf_counter(); r = f(arr, 16); print(f"{getattr(f, '__name__', '?'):<16} {(time.perf_counter()-t)*1000:7.1f} ms")
assert slow_recompute(arr, 16) == Solution().getMaximumXor(arr, 16)

slow_recompute     118.1 ms
getMaximumXor        0.3 ms


**Vì sao không tốt:** cắt lát + `reduce` mỗi truy vấn tốn `O(n)` ⇒ `O(n²)`, với `n = 10⁵` là `5·10⁹`. **Sửa:** cập nhật `X ^= v` (tính chất tự nghịch đảo).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tối đa `X ^ k` với `k < 2^m`" và `X < 2^m` ⇒ **luôn** đạt `2^m − 1` với `k = X ^ mask`.
- "Xóa phần tử cuối rồi hỏi lại" ⇒ duyệt **ngược** + cập nhật dần; với XOR thì bỏ phần tử bằng cách XOR lần nữa.
- Cần `k` có ràng buộc phức tạp hơn (ví dụ `k` thuộc một tập cho trước) ⇒ **Trie nhị phân** (bài 421, 1707).
- Trong Python, tránh `~x` khi cần số không âm — dùng `x ^ mask` hoặc `~x & mask`.
- Chi tiết "mảng đã sắp" ở đây không dùng — đừng bị phân tâm.

## 15. Biến thể của bài

1. **Xóa phần tử đầu** thay vì cuối: duyệt xuôi, cùng công thức.
2. **`k` phải thuộc một mảng `candidates`**: dùng Trie nhị phân trên `candidates`, tham lam từng bit (giống **421 Maximum XOR of Two Numbers in an Array**).
3. **1707 Maximum XOR With an Element From Array**: thêm ràng buộc `k ≤ m_i` — Trie + xử lý truy vấn offline.
4. **Tối thiểu `X ^ k`**: chọn `k = X` ⇒ kết quả 0 (tầm thường); nếu `k` thuộc tập cho trước thì Trie tìm bit **giống**.
5. **1310 XOR Queries of a Subarray**: XOR đoạn `[l, r]` bằng tiền tố `p[r+1] ^ p[l]`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | Tính lại XOR, thử mọi `k` | `O(n(n + 2^m))` | `O(n)` | Kiểm chứng, `n, m` nhỏ |
| 2. Từng bit | Đảo bit 0 của `X` thành 1 cho `k` | `O(n·m)` | `O(1)` | Khi `k` bị ràng buộc theo bit |
| 3. `X ^ mask` | Đảo `m` bit một lần | `O(n)` | `O(1)` | **Bản nộp** |
| 4. Tiền tố | `p[i] ^ mask` đọc ngược | `O(n)` | `O(n)` | Code một dòng, nhiều truy vấn tùy ý |
| 5. Trie nhị phân | Tham lam theo bit trên tập ứng viên | `O(2^m m + n m)` | `O(2^m m)` | Khi `k` bị giới hạn trong một tập |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `([3], 2)`; (b) `([0, 0], 1)`; (c) `([1, 2, 3], 2)`; (d) `([4, 4, 4, 4], 3)`.

**Bài 2 (tính tay):** Với `nums = [1, 2, 5, 6]`, `maximumBit = 3`: lập bảng `X`, `k` (nhị phân) cho cả 4 truy vấn.

**Bài 3 (điền TODO):** Hoàn thiện `best_k(X, m)` trả `k` bằng vòng lặp bit (không dùng `^ mask`).

**Bài 4 (tự viết lại):** Viết `max_xor_remove_first(nums, m)` cho biến thể **xóa phần tử đầu** sau mỗi truy vấn.

**Bài 5 (mini challenge):** Viết `max_xor_from_set(nums, cands)`: với mỗi truy vấn (xóa cuối như đề), chọn `k` **thuộc mảng `cands`** để `X ^ k` lớn nhất, trả các `k` (nếu hòa chọn `k` nhỏ nhất). Dùng **Trie nhị phân** 20 bit, đối chiếu vét cạn.

In [26]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def best_k_todo(X, m):
    k = 0
    for i in range(m):
        # TODO: nếu bit i của X bằng 0 thì bật bit i của k
        pass
    return k

In [27]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def max_xor_remove_first(nums, m):
    # TODO: X = XOR cả mảng; duyệt xuôi: ghi X ^ mask rồi X ^= v
    pass

def max_xor_from_set(nums, cands):
    # TODO: dựng Trie 20 bit cho cands (lưu giá trị nhỏ nhất ở lá);
    # với mỗi X: đi từ bit 19 xuống 0, ưu tiên nhánh có bit ngược với X
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: `mask`. Mức 2: XOR tiền tố đọc ngược. Mức 3: (a) `[0]`; (b) `[1, 1]`; (c) `X`: `0, 3, 1` ⇒ `[3, 0, 2]`; (d) `X`: `0, 4, 0, 4` ⇒ `[7, 3, 7, 3]`.

**Bài 2.** Mức 1: XOR cả mảng `1^2^5^6 = 0`. Mức 2: xóa 6 ⇒ `6`; xóa 5 ⇒ `3`; xóa 2 ⇒ `1`. Mức 3: `k = 7, 1, 4, 6`.

**Bài 3.** Mức 1: `X >> i & 1`. Mức 2: `k |= 1 << i`. Mức 3: so với `X ^ ((1 << m) − 1)` trên mọi `X < 2^m`.

**Bài 4.** Mức 1: giống Cách 3. Mức 2: duyệt `for v in nums`. Mức 3: đối chiếu vét cạn xóa `arr.pop(0)`.

**Bài 5.** Mức 1: Trie dạng dict lồng hoặc mảng `[con0, con1]`. Mức 2: muốn bit kết quả là 1 ⇒ đi nhánh `1 − bit(X)` nếu có. Mức 3: hòa xảy ra khi `cands` có phần tử trùng — lưu giá trị ở lá là duy nhất nên không hòa thật sự; vẫn đối chiếu với `max(cands, key=lambda k: (X ^ k, -k))`.

## 19. Đáp án tham khảo cho bài tập

In [28]:
# Bài 1, 2, 3
b1 = [Solution().getMaximumXor(n, m) for n, m in [([3], 2), ([0, 0], 1), ([1, 2, 3], 2), ([4, 4, 4, 4], 3)]]
print("Bài 1:", b1); assert b1 == [[0], [1, 1], [3, 0, 2], [7, 3, 7, 3]]
print("Bài 2:"); assert trace_mask([1, 2, 5, 6], 3) == [7, 1, 4, 6]
def best_k_todo(X, m):
    k = 0
    for i in range(m):
        if not (X >> i) & 1:
            k |= 1 << i
    return k
assert all(best_k_todo(X, m) == X ^ ((1 << m) - 1) for m in range(1, 9) for X in range(1 << m))
print("Bài 3: OK")

Bài 1: [[0], [1, 1], [3, 0, 2], [7, 3, 7, 3]]
Bài 2:
mask = 111, X ban đầu = 000
q=0: mảng=[1, 2, 5, 6], X=000 (0), k=111 (7), X^k=111 | xóa 6 -> X mới=110
q=1: mảng=[1, 2, 5], X=110 (6), k=001 (1), X^k=111 | xóa 5 -> X mới=011
q=2: mảng=[1, 2], X=011 (3), k=100 (4), X^k=111 | xóa 2 -> X mới=001
q=3: mảng=[1], X=001 (1), k=110 (6), X^k=111 | xóa 1 -> X mới=000
Bài 3: OK


In [29]:
# Bài 4, 5
def max_xor_remove_first(nums, m):
    mask = (1 << m) - 1; X = reduce(xor, nums); ans = []
    for v in nums:
        ans.append(X ^ mask); X ^= v
    return ans

def max_xor_from_set(nums, cands, B=20):
    root = {}
    for c in sorted(set(cands)):
        node = root
        for i in range(B - 1, -1, -1):
            node = node.setdefault(c >> i & 1, {})
        node.setdefault("val", c)
    X = reduce(xor, nums); ans = []
    for v in reversed(nums):
        node = root
        for i in range(B - 1, -1, -1):
            want = 1 - (X >> i & 1)
            node = node[want] if want in node else node[1 - want]
        ans.append(node["val"]); X ^= v
    return ans

rng = random.Random(5)
for _ in range(300):
    m = rng.randint(1, 6); nums = sorted(rng.randrange(1 << m) for _ in range(rng.randint(1, 10)))
    arr, exp4 = list(nums), []
    while arr:
        X = reduce(xor, arr); exp4.append(max(range(1 << m), key=lambda k: X ^ k)); arr.pop(0)
    assert max_xor_remove_first(nums, m) == exp4
    cands = [rng.randrange(1 << m) for _ in range(rng.randint(1, 6))]
    arr, exp5 = list(nums), []
    while arr:
        X = reduce(xor, arr); exp5.append(max(cands, key=lambda k: (X ^ k, -k))); arr.pop()
    assert max_xor_from_set(nums, cands) == exp5
print("Bài 4, 5: OK |", max_xor_remove_first([0, 1, 1, 3], 2), max_xor_from_set([0, 1, 1, 3], [1, 2]))

Bài 4, 5: OK | [0, 0, 1, 0] [1, 2, 2, 2]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
mask = (1 << maximumBit) - 1
X = 0
for v in nums: X ^= v
ans = []
for v in reversed(nums):
    ans.append(X ^ mask)     # k làm X ^ k = mask (lớn nhất)
    X ^= v                   # xóa phần tử cuối khỏi tổng XOR
return ans
```

**Checklist:**
- [ ] Trả **`k`**, không phải giá trị XOR lớn nhất?
- [ ] Dùng `X ^ mask` (không dùng `~X` — ra số âm)?
- [ ] Ghi đáp án **trước**, rồi mới xóa phần tử cuối (`X ^= v`)?
- [ ] Duyệt **ngược** (xóa phần tử cuối)?
- [ ] Đã test `n = 1`, toàn 0, toàn `mask`, `maximumBit = 1` và `20`?